In [1]:
import cv2
import numpy as np
import torch
import torchvision.transforms as T
import torchvision.models as models
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamPredictor
import os

# ========================
# CONFIG
# ========================
YOLO_MODEL_PATH = r"C:\Users\TAI_AI\Desktop\YOLO\runs\detect\pothole_detection\exp12\weights\best.pt"
VIDEO_PATH = r"test/video/istockphoto-937868038-640_adpp_is.mp4"

SAM_CHECKPOINT = "sam_vit_b.pth"

DIST_THRESHOLD = 80
MIN_AREA = 500

SAVE_DIR = "pothole_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# ========================
# LOAD MODEL
# ========================
device = "cuda" if torch.cuda.is_available() else "cpu"

yolo = YOLO(YOLO_MODEL_PATH)

resnet = models.resnet18(pretrained=True)
resnet.fc = torch.nn.Identity()
resnet.to(device)
resnet.eval()

transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor()
])

# ========================
# LOAD SAM
# ========================
SAM_CHECKPOINT = r"C:\Users\TAI_AI\Desktop\YOLO\weights\sam_vit_h_4b8939.pth"

sam = sam_model_registry["vit_h"](checkpoint=SAM_CHECKPOINT)
sam.to(device)
predictor = SamPredictor(sam)

# ========================
# TRACKING STORAGE
# ========================
tracks = []
next_id = 0
saved_ids = set()

class Track:
    def __init__(self, center, feature):
        global next_id
        self.id = next_id
        next_id += 1

        self.center = center
        self.feature = feature
        self.missed = 0

    def update(self, center, feature):
        self.center = center
        self.feature = feature
        self.missed = 0

# ========================
# HELPER FUNCTIONS
# ========================
def get_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1+x2)/2, (y1+y2)/2])

def distance(c1, c2):
    return np.linalg.norm(c1 - c2)

def extract_feature(img):
    img = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = resnet(img)
    return feat / feat.norm()

def cosine_sim(f1, f2):
    return torch.dot(f1.squeeze(), f2.squeeze()).item()

# ========================
# SAM + SAVE FUNCTION
# ========================
def segment_and_save(frame, box, obj_id, frame_id):
    x1, y1, x2, y2 = box

    predictor.set_image(frame)

    input_box = np.array([x1, y1, x2, y2])

    masks, _, _ = predictor.predict(
        box=input_box,
        multimask_output=False
    )

    mask = masks[0]

    # tìm contour
    mask_uint8 = (mask * 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    vis = frame.copy()

    # vẽ contour
    cv2.drawContours(vis, contours, -1, (0, 0, 255), 2)

    # vẽ ID
    cv2.putText(vis, f"ID {obj_id}", (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    crop = vis[y1:y2, x1:x2]

    filename = f"{SAVE_DIR}/id{obj_id}_frame{frame_id}.png"
    cv2.imwrite(filename, crop)

# ========================
# VIDEO
# ========================
cap = cv2.VideoCapture(VIDEO_PATH)

frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    detections = []

    # YOLO detect
    results = yolo(frame)[0]

    for box in results.boxes.xyxy:
        x1, y1, x2, y2 = map(int, box)

        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        area = (x2 - x1) * (y2 - y1)
        if area < MIN_AREA:
            continue

        feature = extract_feature(crop)
        center = get_center([x1, y1, x2, y2])

        detections.append({
            "box": (x1, y1, x2, y2),
            "center": center,
            "feature": feature
        })

    # ========================
    # MATCHING
    # ========================
    for det in detections:
        best_track = None
        best_score = 999

        for track in tracks:
            d = distance(det["center"], track.center)
            sim = cosine_sim(det["feature"], track.feature)

            score = d + (1 - sim) * 100

            if score < best_score:
                best_score = score
                best_track = track

        if best_score < DIST_THRESHOLD:
            best_track.update(det["center"], det["feature"])
            det["id"] = best_track.id
        else:
            new_track = Track(det["center"], det["feature"])
            tracks.append(new_track)
            det["id"] = new_track.id

    # ========================
    # DRAW + SAM
    # ========================
    for det in detections:
        x1, y1, x2, y2 = det["box"]
        id_ = det["id"]

        # vẽ bbox
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, f"ID {id_}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        # chạy SAM chỉ khi chưa lưu
        if id_ not in saved_ids and frame_id % 10 == 0:
            segment_and_save(frame, (x1, y1, x2, y2), id_, frame_id)
            saved_ids.add(id_)

    cv2.imshow("Tracking + SAM", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

C:\Users\TAI_AI\AppData\Roaming\Python\Python310\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\TAI_AI\AppData\Roaming\Python\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



0: 384x640 (no detections), 77.7ms
Speed: 2.3ms preprocess, 77.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 9.7ms
Speed: 2.1ms preprocess, 9.7ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 9.8ms
Speed: 1.4ms preprocess, 9.8ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 16.4ms
Speed: 2.0ms preprocess, 16.4ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 pothole, 10.4ms
Speed: 1.4ms preprocess, 10.4ms inference, 14.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 14.6ms
Speed: 2.3ms preprocess, 14.6ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 10.0ms
Speed: 1.9ms preprocess, 10.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 pothole, 10.4ms
Speed: 1.6ms preprocess, 10.4ms inference, 1.7ms

In [3]:
import cv2
import numpy as np
import torch
import torchvision.transforms as T
import torchvision.models as models
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamPredictor
import os

# ========================
# CONFIG
# ========================
YOLO_MODEL_PATH = r"C:\Users\TAI_AI\Desktop\YOLO\runs\detect\pothole_detection\optimized_model_v22\weights\best.pt"
VIDEO_PATH = r"test/video/istockphoto-937868038-640_adpp_is.mp4"

SAM_CHECKPOINT = "sam_vit_b.pth"

DIST_THRESHOLD = 80
MIN_AREA = 500

SAVE_DIR = "pothole_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# ========================
# LOAD MODEL
# ========================
device = "cuda" if torch.cuda.is_available() else "cpu"

yolo = YOLO(YOLO_MODEL_PATH)

resnet = models.resnet18(pretrained=True)
resnet.fc = torch.nn.Identity()
resnet.to(device)
resnet.eval()

transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor()
])

# ========================
# LOAD SAM
# ========================
SAM_CHECKPOINT = r"C:\Users\TAI_AI\Desktop\YOLO\weights\sam_vit_h_4b8939.pth"

sam = sam_model_registry["vit_h"](checkpoint=SAM_CHECKPOINT)
sam.to(device)
predictor = SamPredictor(sam)

# ========================
# TRACKING STORAGE
# ========================
tracks = []
next_id = 0
saved_ids = set()

class Track:
    def __init__(self, center, feature):
        global next_id
        self.id = next_id
        next_id += 1

        self.center = center
        self.feature = feature
        self.missed = 0

    def update(self, center, feature):
        self.center = center
        self.feature = feature
        self.missed = 0

# ========================
# HELPER FUNCTIONS
# ========================
def get_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1+x2)/2, (y1+y2)/2])

def distance(c1, c2):
    return np.linalg.norm(c1 - c2)

def extract_feature(img):
    img = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = resnet(img)
    return feat / feat.norm()

def cosine_sim(f1, f2):
    return torch.dot(f1.squeeze(), f2.squeeze()).item()

# ========================
# SAM + SAVE FUNCTION
# ========================
def segment_and_save(frame, box, obj_id, frame_id):
    x1, y1, x2, y2 = box

    predictor.set_image(frame)

    input_box = np.array([x1, y1, x2, y2])

    masks, _, _ = predictor.predict(
        box=input_box,
        multimask_output=False
    )

    mask = masks[0]

    # tìm contour
    mask_uint8 = (mask * 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    vis = frame.copy()

    # vẽ contour
    cv2.drawContours(vis, contours, -1, (0, 0, 255), 2)

    # vẽ ID
    cv2.putText(vis, f"ID {obj_id}", (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    crop = vis[y1:y2, x1:x2]

    filename = f"{SAVE_DIR}/id{obj_id}_frame{frame_id}.png"
    cv2.imwrite(filename, crop)

# ========================
# VIDEO
# ========================
cap = cv2.VideoCapture(VIDEO_PATH)

frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    detections = []

    # YOLO detect
    results = yolo(frame)[0]

    for box in results.boxes.xyxy:
        x1, y1, x2, y2 = map(int, box)

        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        area = (x2 - x1) * (y2 - y1)
        if area < MIN_AREA:
            continue

        feature = extract_feature(crop)
        center = get_center([x1, y1, x2, y2])

        detections.append({
            "box": (x1, y1, x2, y2),
            "center": center,
            "feature": feature
        })

    # ========================
    # MATCHING
    # ========================
    for det in detections:
        best_track = None
        best_score = 999

        for track in tracks:
            d = distance(det["center"], track.center)
            sim = cosine_sim(det["feature"], track.feature)

            score = d + (1 - sim) * 100

            if score < best_score:
                best_score = score
                best_track = track

        if best_score < DIST_THRESHOLD:
            best_track.update(det["center"], det["feature"])
            det["id"] = best_track.id
        else:
            new_track = Track(det["center"], det["feature"])
            tracks.append(new_track)
            det["id"] = new_track.id

    # ========================
    # DRAW + SAM
    # ========================
    for det in detections:
        x1, y1, x2, y2 = det["box"]
        id_ = det["id"]

        # vẽ bbox
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, f"ID {id_}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        # chạy SAM chỉ khi chưa lưu
        if id_ not in saved_ids and frame_id % 10 == 0:
            segment_and_save(frame, (x1, y1, x2, y2), id_, frame_id)
            saved_ids.add(id_)

    cv2.imshow("Tracking + SAM", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 (no detections), 11.1ms
Speed: 9.2ms preprocess, 11.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 11.3ms
Speed: 1.8ms preprocess, 11.3ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 10.1ms
Speed: 1.7ms preprocess, 10.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 9.8ms
Speed: 1.4ms preprocess, 9.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 pothole, 9.8ms
Speed: 1.4ms preprocess, 9.8ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 9.8ms
Speed: 1.5ms preprocess, 9.8ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 9.9ms
Speed: 1.5ms preprocess, 9.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 9.5ms
Speed: 1.7ms preprocess, 9.5ms inference, 0.7ms 

In [4]:
import cv2
import numpy as np
import torch
import torchvision.transforms as T
import torchvision.models as models
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamPredictor
import os
import subprocess

# ========================
# CONFIG
# ========================
YOLO_MODEL_PATH = r"C:\Users\TAI_AI\Desktop\YOLO\runs\detect\pothole_detection\optimized_model_v22\weights\best.pt"
VIDEO_PATH = r"test/video/istockphoto-937868038-640_adpp_is.mp4"
SAM_CHECKPOINT = r"C:\Users\TAI_AI\Desktop\YOLO\weights\sam_vit_h_4b8939.pth"

DIST_THRESHOLD = 80
MIN_AREA = 500

SAVE_DIR = "pothole_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# Lấy đường dẫn tuyệt đối để thông báo cho người dùng
ABS_SAVE_PATH = os.path.abspath(SAVE_DIR)
print(f"\n[HỆ THỐNG] Dữ liệu sẽ được lưu tại: {ABS_SAVE_PATH}\n")

# ========================
# LOAD MODELS
# ========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Đang chạy trên thiết bị: {device}")

yolo = YOLO(YOLO_MODEL_PATH)

# Feature extractor (ResNet18) cho Tracking
resnet = models.resnet18(weights='IMAGENET1K_V1')
resnet.fc = torch.nn.Identity()
resnet.to(device)
resnet.eval()

transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor()
])

# Load SAM
sam = sam_model_registry["vit_h"](checkpoint=SAM_CHECKPOINT)
sam.to(device)
predictor = SamPredictor(sam)

# ========================
# TRACKING STORAGE
# ========================
tracks = []
next_id = 0
saved_ids = set()

class Track:
    def __init__(self, center, feature):
        global next_id
        self.id = next_id
        next_id += 1
        self.center = center
        self.feature = feature
        self.missed = 0

    def update(self, center, feature):
        self.center = center
        self.feature = feature
        self.missed = 0

# ========================
# HELPER FUNCTIONS
# ========================
def get_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1+x2)/2, (y1+y2)/2])

def distance(c1, c2):
    return np.linalg.norm(c1 - c2)

def extract_feature(img):
    img = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = resnet(img)
    return feat / feat.norm()

def cosine_sim(f1, f2):
    return torch.dot(f1.squeeze(), f2.squeeze()).item()

def segment_and_save(frame, box, obj_id, frame_id):
    x1, y1, x2, y2 = box
    predictor.set_image(frame)
    input_box = np.array([x1, y1, x2, y2])

    masks, _, _ = predictor.predict(
        box=input_box,
        multimask_output=False
    )
    mask = masks[0]

    # Tìm và vẽ contour
    mask_uint8 = (mask * 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    vis = frame.copy()
    cv2.drawContours(vis, contours, -1, (0, 0, 255), 2)
    cv2.putText(vis, f"ID {obj_id}", (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

    crop = vis[y1:y2, x1:x2]
    filename = os.path.join(SAVE_DIR, f"id{obj_id}_frame{frame_id}.png")
    cv2.imwrite(filename, crop)
    print(f" -> Đã lưu ổ gà ID {obj_id} vào thư mục.")

# ========================
# VIDEO PROCESSING
# ========================
cap = cv2.VideoCapture(VIDEO_PATH)
frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    detections = []

    # YOLO detect
    results = yolo(frame, verbose=False)[0]

    for box in results.boxes.xyxy:
        x1, y1, x2, y2 = map(int, box)
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0: continue

        area = (x2 - x1) * (y2 - y1)
        if area < MIN_AREA: continue

        feature = extract_feature(crop)
        center = get_center([x1, y1, x2, y2])

        detections.append({
            "box": (x1, y1, x2, y2),
            "center": center,
            "feature": feature
        })

    # Matching (Tracking)
    for det in detections:
        best_track = None
        best_score = 999

        for track in tracks:
            d = distance(det["center"], track.center)
            sim = cosine_sim(det["feature"], track.feature)
            score = d + (1 - sim) * 100

            if score < best_score:
                best_score = score
                best_track = track

        if best_score < DIST_THRESHOLD:
            best_track.update(det["center"], det["feature"])
            det["id"] = best_track.id
        else:
            new_track = Track(det["center"], det["feature"])
            tracks.append(new_track)
            det["id"] = new_track.id

    # Drawing and Saving
    for det in detections:
        x1, y1, x2, y2 = det["box"]
        id_ = det["id"]

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, f"ID {id_}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        # Chỉ chạy SAM nếu ID này chưa được lưu
        if id_ not in saved_ids:
            segment_and_save(frame, (x1, y1, x2, y2), id_, frame_id)
            saved_ids.add(id_)

    cv2.imshow("Pothole Tracking & SAM", frame)

    if cv2.waitKey(1) & 0xFF == 27: # Nhấn ESC để thoát
        break

cap.release()
cv2.destroyAllWindows()

# Kết thúc: In thông báo và mở thư mục
print("-" * 30)
print(f"Xử lý hoàn tất!")
print(f"Toàn bộ ảnh đã được lưu tại: {ABS_SAVE_PATH}")
print("-" * 30)

# Tự động mở thư mục (Dành cho Windows)
os.startfile(ABS_SAVE_PATH)


[HỆ THỐNG] Dữ liệu sẽ được lưu tại: c:\Users\TAI_AI\Desktop\YOLO\pothole_results

[INFO] Đang chạy trên thiết bị: cuda
 -> Đã lưu ổ gà ID 0 vào thư mục.
 -> Đã lưu ổ gà ID 1 vào thư mục.
 -> Đã lưu ổ gà ID 2 vào thư mục.
 -> Đã lưu ổ gà ID 3 vào thư mục.
 -> Đã lưu ổ gà ID 4 vào thư mục.
 -> Đã lưu ổ gà ID 5 vào thư mục.
 -> Đã lưu ổ gà ID 6 vào thư mục.
 -> Đã lưu ổ gà ID 7 vào thư mục.
 -> Đã lưu ổ gà ID 8 vào thư mục.
 -> Đã lưu ổ gà ID 9 vào thư mục.
 -> Đã lưu ổ gà ID 10 vào thư mục.
 -> Đã lưu ổ gà ID 11 vào thư mục.
 -> Đã lưu ổ gà ID 12 vào thư mục.
 -> Đã lưu ổ gà ID 13 vào thư mục.
 -> Đã lưu ổ gà ID 14 vào thư mục.
 -> Đã lưu ổ gà ID 15 vào thư mục.
 -> Đã lưu ổ gà ID 16 vào thư mục.
 -> Đã lưu ổ gà ID 17 vào thư mục.
 -> Đã lưu ổ gà ID 18 vào thư mục.
 -> Đã lưu ổ gà ID 19 vào thư mục.
 -> Đã lưu ổ gà ID 20 vào thư mục.
 -> Đã lưu ổ gà ID 21 vào thư mục.
 -> Đã lưu ổ gà ID 22 vào thư mục.
 -> Đã lưu ổ gà ID 23 vào thư mục.
 -> Đã lưu ổ gà ID 24 vào thư mục.
 -> Đã lưu ổ gà